In [4]:
#!/usr/bin/env python3

# %pip install -U "transformers<5" datasets evaluate accelerate scikit-learn
# %pip install -U torch torchvision --index-url https://download.pytorch.org/whl/cu126

In [ ]:
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

In [7]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Устройство:", device)
if torch.cuda.is_available():
    print("Видеокарта:", torch.cuda.get_device_name(0))

Устройство: cuda
Видеокарта: NVIDIA GeForce RTX 3050 Laptop GPU


[Датасет отзывов на русском](https://huggingface.co/datasets/angryelizar/sentiment_dataset_splitted), 
[модель RuBert](https://huggingface.co/ai-forever/ruBert-base)

Загрузим датасет

In [12]:
DATASET_NAME = "angryelizar/sentiment_dataset_splitted"

dataset = load_dataset(DATASET_NAME)

display(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'src'],
        num_rows: 232366
    })
    test: Dataset({
        features: ['text', 'label', 'src'],
        num_rows: 58092
    })
})

Кратко осмотрим датасет

In [14]:
display(dataset["train"].features)

{'text': Value('string'),
 'label': ClassLabel(names=['neutral', 'positive', 'negative']),
 'src': Value('string')}

In [16]:
label_names = dataset["train"].features["label"].names
for split_name in dataset:
    print(f"\n{split_name}:")
    counts = pd.Series(dataset[split_name]["label"]).value_counts().sort_index()
    for label, count in counts.items():
        print(f"{label_names[label]}: {count}")


train:
neutral: 77271
positive: 77502
negative: 77593

test:
neutral: 19318
positive: 19375
negative: 19399


Разделим `train` на две части: 90% для обучения и 10% для валидации

In [17]:
split_dataset = dataset["train"].train_test_split(
    test_size=0.1,
    seed=SEED,
    stratify_by_column="label"
)

train_dataset = split_dataset["train"]
valid_dataset = split_dataset["test"]
test_dataset = dataset["test"]

print("Train:", len(train_dataset))
print("Validation:", len(valid_dataset))
print("Test:", len(test_dataset))

Train: 209129
Validation: 23237
Test: 58092


Загрузим токенизатор под RuBERT

In [21]:
MODEL_NAME = "ai-forever/ruBert-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Токенизатор загружен:", MODEL_NAME)
print("Размер словаря:", tokenizer.vocab_size)

Токенизатор загружен: ai-forever/ruBert-base
Размер словаря: 120138


Посмотрим, какой объем токенов опционально использовать

In [ ]:
lengths = []
for i in range(0, len(train_dataset), 256):
    batch = train_dataset[i:i + 256]
    encoded = tokenizer(
        batch["text"],
        truncation=False
    )
    lengths.extend([len(ids) for ids in encoded["input_ids"]])
lengths = np.array(lengths)
print("Средняя длина:", round(lengths.mean(), 1))
print("Медиана:", np.percentile(lengths, 50))
print("90-й процентиль:", np.percentile(lengths, 90))
print("95-й процентиль:", np.percentile(lengths, 95))
print("99-й процентиль:", np.percentile(lengths, 99))
print("Максимум:", lengths.max())
for limit in [128, 256, 384, 512]:
    truncated = (lengths > limit).mean() * 100
    print(f"Длина {limit}: будет обрезано {truncated:.2f}% отзывов")

Средняя длина: 78.8
Медиана: 42.0
90-й процентиль: 218.0
95-й процентиль: 269.0
99-й процентиль: 325.0
Максимум: 916
Длина 128: будет обрезано 21.94% отзывов
Длина 256: будет обрезано 6.11% отзывов
Длина 384: будет обрезано 0.04% отзывов
Длина 512: будет обрезано 0.00% отзывов


Зададим функцию токенизации

In [27]:
MAX_LENGTH = 512

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_LENGTH
    )

Токенезируем датасет

In [30]:
tokenized_train = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text", "src"],
    desc="Tokenizing train"
)

tokenized_valid = valid_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text", "src"],
    desc="Tokenizing validation"
)

tokenized_test = test_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text", "src"],
    desc="Tokenizing test"
)

In [35]:
print("Столбцы:", tokenized_train.column_names)

print("\nРазмеры выборок:")
print("Train:", len(tokenized_train))
print("Validation:", len(tokenized_valid))
print("Test:", len(tokenized_test)) 

Столбцы: ['label', 'input_ids', 'token_type_ids', 'attention_mask']

Размеры выборок:
Train: 209129
Validation: 23237
Test: 58092


In [38]:
print("\nПример токенизированных данных:")
print(tokenized_train[0])


Пример токенизированных данных:
{'label': 1, 'input_ids': [101, 32326, 378, 41080, 113, 1303, 660, 42307, 750, 107, 654, 33963, 5404, 780, 2184, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


Подготовим collator, который будет выравнивать длину отзывов в батчах

In [32]:
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

Загружаем предобученный RuBERT и добавляем классификационный слой, который будет определять тональность отзыва

In [40]:
id2label = {
    0: "neutral",
    1: "positive",
    2: "negative"
}

label2id = {
    name: idx for idx, name in id2label.items()
}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

print("Модель загружена:", MODEL_NAME)
print("Количество классов:", model.config.num_labels)
print("Количество параметров:", sum(p.numel() for p in model.parameters()))

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at ai-forever/ruBert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Модель загружена: ai-forever/ruBert-base
Количество классов: 3
Количество параметров: 178309635


Настроим метрики

In [41]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1_macro": f1_score(
            labels,
            predictions,
            average="macro"
        )
    }

Настроим обучение

In [43]:
training_args = TrainingArguments(
    output_dir="./rubert_results",

    # Продолжительность обучения
    num_train_epochs=1,

    # Размер батча
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,

    # Оптимизация
    learning_rate=2e-5,
    weight_decay=0.01,
    optim="adafactor",

    # Экономия видеопамяти
    fp16=True,
    gradient_checkpointing=True,

    # Оценка качества
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_f1_macro",
    greater_is_better=True,

    # Логирование
    logging_steps=100,
    save_total_limit=1,
    eval_accumulation_steps=16,

    seed=SEED,
    report_to="none",
    dataloader_num_workers=0
)

Создаем Trainer

In [44]:
trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=tokenized_train,
    eval_dataset=tokenized_valid,

    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("Trainer готов к обучению")

Trainer готов к обучению


In [ ]:
# train_result = trainer.train()

# trainer.save_model("./rubert_sentiment_model")
# tokenizer.save_pretrained("./rubert_sentiment_model")

# print("Модель и токенизатор сохранены!")

Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 